# Barter Python: Engine & Backtesting

This notebook covers the engine pipeline: loading configs, running backtests,
and writing custom strategy, risk management, and event callback logic in Python.

## Topics Covered
- Loading `SystemConfig` from JSON and historical market data
- Running a backtest with `run_backtest()`
- Writing a custom strategy callback with `EngineState` and `OrderRequestOpen`
- Writing a custom risk management callback
- Multi-strategy composition with `strategies={"name": fn, ...}`
- Event callbacks: `on_fill` and `on_position_closed`
- Inspecting backtest results via `TradingSummary`

## Prerequisites

These notebooks assume `barter` is installed into the active IPython kernel, for example:

```bash
cd /mnt/developer/git/aecs4u.it/quant/barter-rs/barter-python
uv sync --dev
source .venv/bin/activate
python -m pip install -e .
```

If you register a dedicated kernel, select that kernel before running the notebook.


In [ ]:
from pathlib import Path
import sys

def find_repo_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        python_dir = candidate / "barter-python" / "python" / "barter"
        if (python_dir / "__init__.py").exists():
            return candidate
    raise RuntimeError("Could not locate the barter-rs repository root from the current working directory.")

REPO_ROOT = find_repo_root()
PYTHON_SOURCE = REPO_ROOT / "barter-python" / "python"
if str(PYTHON_SOURCE) not in sys.path:
    sys.path.insert(0, str(PYTHON_SOURCE))

print(f"Using barter from {PYTHON_SOURCE}")


In [ ]:
import json

import barter
from barter import Balance, OrderRequestOpen, OrderRequestCancel, EngineState, run_backtest

---
## Part 1: Engine & Backtesting


## 1. Load Example Config and Historical Market Data

The repo already contains a `SystemConfig` example and matching historical trade events.

In [ ]:
config_path = REPO_ROOT / "barter" / "examples" / "config" / "backtest_config.json"
market_data_path = REPO_ROOT / "barter" / "examples" / "data" / "binance_spot_market_data_with_disconnect_events.json"

example_config = json.loads(config_path.read_text())
system_config_json = json.dumps(example_config["system"])
market_data_json = market_data_path.read_text()

print(config_path)
print(market_data_path)

## 2. Run the Backtest

The current Python API uses the built-in noop strategy and noop risk manager wired into `run_backtest()`. That makes this a baseline engine integration example rather than a strategy notebook.

In [ ]:
summary = await run_backtest(
    config_json=system_config_json,
    market_data_json=market_data_json,
    risk_free_return=example_config.get("risk_free_return", 0.05),
)

summary_dict = summary.to_dict()
print(summary)
print(sorted(summary_dict.keys()))

## 3. Inspect the Result

The summary payload is returned as a Python object with both `to_json()` and `to_dict()` helpers.

In [ ]:
print(summary.to_json()[:4000])

---
## Part 2: Custom Strategy Callback

The `run_backtest()` function accepts an optional `strategy=` argument — a
Python callable that receives an `EngineState` snapshot on every engine tick
and returns a list of `OrderRequestOpen` (or a tuple of cancels and opens).

The `EngineState` snapshot provides:
- `state.price(instrument_index)` — current market price
- `state.position(instrument_index)` — current position (side, qty, entry price, unrealised PnL)
- `state.instruments()` — list of all instrument dicts
- `state.balances()` — list of all asset balance dicts
- `state.balance(asset_name)` — balance for a specific asset

In [ ]:
# A simple strategy that buys BTC on every tick where a price is available
def buy_btc_strategy(state):
    """Buy 0.001 BTC whenever a price is available for instrument 0."""
    orders = []
    price = state.price(0)  # instrument index 0 = BTCUSDT
    if price is not None:
        orders.append(
            OrderRequestOpen(
                exchange_index=0,
                instrument_index=0,
                side="buy",
                price=price,
                quantity=0.001,
                order_kind="market",   # "market" or "limit"
                time_in_force="ioc",   # "ioc", "gtc", "fok", "gtd"
            )
        )
    return orders

# Run with the custom strategy
strategy_summary = await run_backtest(
    config_json=system_config_json,
    market_data_json=market_data_json,
    risk_free_return=example_config.get("risk_free_return", 0.05),
    strategy=buy_btc_strategy,
)

print(strategy_summary)

## Inspecting the EngineState

The strategy callback receives an `EngineState` snapshot on every tick.
Here's what it looks like:

In [ ]:
---
## Part 3: Custom Risk Management

The `run_backtest()` function also accepts an optional `risk=` argument — a
Python callable that receives `(state, orders)` and returns the subset of
orders that should be approved.

- `state` is the same `EngineState` snapshot
- `orders` is the `List[OrderRequestOpen]` proposed by the strategy
- Return only the orders you want to execute; omitted orders are rejected

# Risk manager: reject orders with notional value > $100
rejected_count = 0

def max_notional_risk(state, orders):
    """Approve only orders whose notional (price * qty) is under $100."""
    global rejected_count
    approved = []
    for order in orders:
        notional = order.price * order.quantity
        if notional <= 100.0:
            approved.append(order)
        else:
            rejected_count += 1
    return approved

# Strategy that produces both small and large orders
def mixed_orders_strategy(state):
    price = state.price(0)
    if price is None:
        return []
    return [
        OrderRequestOpen(0, 0, "buy", price, 0.001),   # small: ~$97
        OrderRequestOpen(0, 0, "buy", price, 0.01),    # large: ~$970 → rejected
    ]

risk_summary = await run_backtest(
    config_json=system_config_json,
    market_data_json=market_data_json,
    strategy=mixed_orders_strategy,
    risk=max_notional_risk,
)

print(f"Result: {risk_summary}")
print(f"Orders rejected by risk manager: {rejected_count}")

In [ ]:
---
## Part 4: Multi-Strategy Composition

In [ ]:
# Multiple named strategies, each with independent position tracking.
# Pass strategies= as a dict of {name: callback}.

def momentum_strategy(state):
    """Buy when price is available."""
    price = state.price(0)
    if price:
        return [OrderRequestOpen(0, 0, "buy", price, 0.001, strategy_id="momentum")]
    return []

def mean_reversion_strategy(state):
    """Sell when price is available."""
    price = state.price(0)
    if price:
        return [OrderRequestOpen(0, 0, "sell", price, 0.0005, strategy_id="mean_rev")]
    return []

multi_summary = await run_backtest(
    config_json=system_config_json,
    market_data_json=market_data_json,
    strategies={
        "momentum": momentum_strategy,
        "mean_rev": mean_reversion_strategy,
    },
)
print(multi_summary)

---
## Part 5: Event Callbacks

Use `on_fill=` to observe every trade fill, and `on_position_closed=` to observe
when a strategy's position is fully exited.

In [ ]:
# on_fill fires on every trade fill during the backtest
fill_log = []

def on_fill(trade):
    fill_log.append(f"{trade.side} {trade.quantity} @ {trade.price:.2f} (strategy={trade.strategy_id})")

# on_position_closed fires when a position is fully exited
closed_log = []

def on_position_closed(pos):
    closed_log.append(f"instrument={pos.instrument_index} PnL={pos.pnl_realised:.2f}")

callback_summary = await run_backtest(
    config_json=system_config_json,
    market_data_json=market_data_json,
    strategy=buy_btc_strategy,
    on_fill=on_fill,
    on_position_closed=on_position_closed,
)

print(f"Fills observed: {len(fill_log)}")
for f in fill_log[:5]:
    print(f"  {f}")

print(f"Positions closed: {len(closed_log)}")
for c in closed_log[:5]:
    print(f"  {c}")

---
## Part 6: Balance Type and API Summary

In [ ]:
# The Balance type represents wallet state
wallet = Balance(1000, 850)
print(wallet)
print({
    "total": wallet.total_f64,
    "free": wallet.free_f64,
    "locked": float(wallet.locked),
})

## API Summary

| Function / Type | Purpose |
|---|---|
| `run_backtest(config, data, ...)` | Run a full backtest with optional callbacks |
| `strategy=fn` | Single strategy callback |
| `strategies={"name": fn, ...}` | Multi-strategy with per-strategy position tracking |
| `risk=fn` | Risk management callback |
| `on_fill=fn` | Called on every trade fill |
| `on_position_closed=fn` | Called when a position is fully exited |
| `OrderRequestOpen(exchange, instrument, side, price, qty)` | Construct an order |
| `OrderRequestCancel(exchange, instrument)` | Construct a cancellation |
| `EngineState` | Read-only snapshot passed to callbacks each tick |
| `EngineState.price(idx)` | Current market price for an instrument |
| `EngineState.position(idx)` | Current position (side, qty, entry, PnL) or None |
| `EngineState.open_orders(idx)` | Count of active orders for an instrument |
| `EngineState.instruments()` | List of all instrument state dicts |
| `EngineState.balances()` | List of all asset balance dicts |
| `TradeFill` | Fill event: trade_id, side, price, quantity, fees, strategy_id |
| `PositionExited` | Closed position: pnl_realised, entry_price, quantity, timestamps |
| `Balance(total, free)` | Wallet balance with locked amount calculation |
| `TradingSummary` | Backtest result with `.to_dict()` and `.to_json()` |

### What Is Still Rust-Only

- `OnDisconnectStrategy` and `OnTradingDisabled` callbacks
- Live trading with real exchange execution (`run_live()` — blocked on upstream execution client)
- L2 order book depth in strategy callbacks (available via `build_order_book_manager()`)